In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pyarrow as pa
import pyarrow.compute as pc
import random

# Change font to respect submission requirements
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["font.weight"] = "bold"

# srun -w mauao -c 5 --partition=interactive --pty bash
# srun -w ngongotaha -c 5 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
# ssh -L 8889:localhost:8889 ls985@ngongotaha

In [ ]:
RESULTS_DIR = Path("/nfs-share/ls985/pollen_worker/results")

In [ ]:
FRAMEWORKS = ["Pollen", "Parrot", "Flower", "FedScale", "Flute"]  # New labels

In [ ]:
DATASETS = ["TG", "IC", "SR", "MLM"]

In [ ]:
DATASETS_FLWR = {
    "openimage": "IC",
    "google_speech": "SR",
    "shakespeare_memory": "TG",
    "reddit": "MLM",
}

DATASETS_FEDSCALE = {
    "openimage": "IC",
    "google_speech": "SR",
    "shakespeare": "TG",
    "reddit": "MLM",
}

DATASETS_FLUTE = {
    "openImg": "IC",
    "google_speech": "SR",
    "shakespeare": "TG",
    "reddit": "MLM",
}

In [ ]:
HARDWARE_SETTINGS = {
    1: "1 GPU",
    2: "1+3 GPUs",
}

In [ ]:
N_TOTAL_ROUNDS = 5_000

In [ ]:
SCALES = {
    "IC": {
        100: "Medium Scale",
        1_000: "Large Scale",
        10_000: "Very Large Scale",
    },
    "SR": {
        100: "Medium Scale",
        1_000: "Large Scale",
        2_000: "Very Large Scale",
    },
    "TG": {
        100: "Medium Scale",
        1_000: "Large Scale",
        10_000: "Very Large Scale",
    },
    "MLM": {
        100: "Medium Scale",
        1_000: "Large Scale",
    },
}

In [ ]:
def error_bar_fn_std_based(x) -> tuple[float, float]:
    return (
        np.sum(x) - np.sqrt(N_TOTAL_ROUNDS * (5 * np.std(x)) ** 2),
        np.sum(x) + np.sqrt(N_TOTAL_ROUNDS * (5 * np.std(x)) ** 2),
    )

In [ ]:
def error_bar_fn_min_max(x) -> tuple[float, float]:
    return (N_TOTAL_ROUNDS * np.min(x), N_TOTAL_ROUNDS * np.max(x))

In [ ]:
def read_logs_flwr(log_file_path: Path) -> pa.Table | None:
    """Reads the log file and returns a pandas dataframe with the data."""
    # Init data for the table
    hardware_setting = 0  # 1 or 2
    n_clients_per_round = 0
    round_numbers: list[int] = []
    start_timestamp: list[pd.Timestamp] = []
    end_timestamp: list[pd.Timestamp] = []
    previous_timestamp: pd.Timestamp | None = None
    # Opening log file
    with open(log_file_path) as f:
        lines = f.readlines()
    while previous_timestamp is None:
        line = lines.pop(0)
        if line.startswith("[") and "global_state_accessor.cc:356:" not in line.split(
            " "
        ):
            strings = line.split(" ")
            while "" in strings:
                strings.remove("")
            # Reading first timestamp
            date = strings[0].replace("[", "")
            time = strings[1].split("]")[0]
            previous_timestamp = pd.to_datetime(
                f"{date} {time}", format="%Y-%m-%d %H:%M:%S,%f"
            )
            # Reading dataset
            dataset = DATASETS_FLWR[strings[5]]
            # Reading framework
            framework = (
                "Flower"
                if log_file_path.name.startswith("R")
                else "Pollen-{}".format(
                    strings[13].split("=")[1].upper().replace("\n", "")
                )
            )
            framework = "Parrot" if framework == "Pollen-LLB" else framework
            framework = "Pollen" if framework == "Pollen-LB" else framework
    # Reading log file
    cnt = 0
    for line in lines:
        if line.startswith("["):
            strings = line.split(" ")
            while "" in strings:
                strings.remove("")

            if hardware_setting == 0:
                if framework == "Flower" and "'GPU':" in strings:
                    idx = strings.index("'GPU':")
                    hardware_setting = (
                        1 if int(strings[idx + 1].split(".")[0]) == 1 else 2
                    )
                elif framework.startswith("P") and (
                    "sending" in strings and "instructions" in strings
                ):
                    hardware_setting = int(strings[8])

            if ("fit_round" in strings) and ("strategy" in strings):
                date = strings[0].replace("[", "")
                time = strings[1].split("]")[0]
                last_timestamp = pd.to_datetime(
                    f"{date} {time}", format="%Y-%m-%d %H:%M:%S,%f"
                )
                if n_clients_per_round == 0:
                    n_clients_per_round = int(strings[7])
                round_numbers.append(cnt)
                cnt += 1
                start_timestamp.append(previous_timestamp)
                previous_timestamp = last_timestamp
                end_timestamp.append(last_timestamp)
    # Creating dict to pass to pa.Table builder
    dict_for_table = {
        "rct": [
            (t1 - t0) for t0, t1 in zip(start_timestamp, end_timestamp, strict=False)
        ],
        "round": round_numbers,
        "n_clients_per_round": [n_clients_per_round] * len(round_numbers),
        "dataset": [dataset] * len(round_numbers),
        "framework": [framework] * len(round_numbers),
        "hardware_setting": [hardware_setting] * len(round_numbers),
        "is_dropped": [False] * len(round_numbers),
    }
    # Returning pa.Table
    return pa.Table.from_pydict(dict_for_table)

In [ ]:
def read_logs_fedscale(log_file_path: Path) -> pa.Table | None:
    """Reads the log file and returns a pandas dataframe with the data."""
    # Init data for the table
    hardware_setting = 0  # 1 or 2
    n_clients_per_round = 0
    round_numbers: list[int] = []
    start_timestamp: list[pd.Timestamp] = []
    end_timestamp: list[pd.Timestamp] = []
    is_dropped: list[bool] = []
    previous_timestamp: pd.Timestamp | None = None
    # Opening log file
    with open(log_file_path) as f:
        lines = f.readlines()
    # Reading first configuration line
    line = lines.pop(0)
    strings = line.split(" ")
    while "" in strings:
        strings.remove("")
    # Reading first timestamp
    date = strings[0]  # format is "(mm-dd)"
    time = strings[1]  # format is "hh:mm:ss"
    previous_timestamp = pd.to_datetime(
        f"(2023-{date}) {time}", format="(%Y-(%m-%d)) %H:%M:%S"
    )
    # Reading dataset
    dataset = DATASETS_FEDSCALE[strings[6].split("'")[1].split("-")[0]]
    # Reading framework
    framework = "FedScale"
    # Reading the hardware setting
    hardware_setting = len(strings[16].split("'")[1].split("="))
    # Reading log file
    cnt = 0
    for line in lines:
        to_be_discarded = False
        strings = line.split(" ")
        while "" in strings:
            strings.remove("")
        if "morons." in strings:
            to_be_discarded = True
        if (
            ("Empty" in strings)
            and ("results" in strings)
            and ("from" in strings)
            and ("client" in strings)
        ):
            to_be_discarded = True
        if "[aggregator.py:543]" in strings and "round:" in strings:
            date = strings[0]  # format is "(mm-dd)"
            time = strings[1]  # format is "hh:mm:ss"
            last_timestamp = pd.to_datetime(
                f"(2023-{date}) {time}", format="(%Y-(%m-%d)) %H:%M:%S"
            )
            if to_be_discarded:
                print(
                    f"Round {cnt} to be discarded due to morons or failed"
                    " deserilization"
                )
            if n_clients_per_round == 0:
                n_clients_per_round = int(strings[15].replace(",", ""))
            to_be_discarded = False
            round_numbers.append(cnt)
            cnt += 1
            start_timestamp.append(previous_timestamp)
            previous_timestamp = last_timestamp
            end_timestamp.append(last_timestamp)
            is_dropped.append(to_be_discarded)
    # Creating dict to pass to pa.Table builder
    dict_for_table = {
        "rct": [
            (t1 - t0) for t0, t1 in zip(start_timestamp, end_timestamp, strict=False)
        ],
        "round": round_numbers,
        "n_clients_per_round": [n_clients_per_round] * len(round_numbers),
        "dataset": [dataset] * len(round_numbers),
        "framework": [framework] * len(round_numbers),
        "hardware_setting": [hardware_setting] * len(round_numbers),
        "is_dropped": is_dropped,
    }
    # Returning pa.Table
    return pa.Table.from_pydict(dict_for_table)

In [ ]:
def read_logs_flute(log_file_path: Path):
    """Reads the log file and returns a pandas dataframe with the data."""
    # Init data for the table
    n_clients_per_round = 0
    round_numbers: list[int] = []
    start_timestamp: list[pd.Timestamp] = []
    end_timestamp: list[pd.Timestamp] = []
    previous_timestamp: pd.Timestamp | None = None
    # Opening log file
    with open(log_file_path) as f:
        lines = f.readlines()
    # Reading the hardware setting
    line = lines.pop(3)
    hardware_setting = 1 if int(line[-2]) == 1 else 2
    # Reading dataset
    line = lines.pop(4)
    strings = line.split(" ")
    dataset = DATASETS_FLUTE[strings[5].replace("'", "").replace(",", "")]
    # Reading framework
    framework = "Flute"
    # Reading the first timestamp
    while previous_timestamp is None:
        line = lines.pop(0)
        strings = line.split(" ")
        while "" in strings:
            strings.remove("")
        if "Assigning" in strings and "default" in strings and "values" in strings:
            datestring = line.split(" : ")[0].replace(" ", "-")
            previous_timestamp = pd.to_datetime(
                datestring, format="%a-%b-%d-%H:%M:%S-%Y"
            )
    # Reading log file
    cnt = 0
    for line in lines:
        strings = line.split(" ")
        if n_clients_per_round == 0:
            if "Clients" in strings and "for" in strings and "round" in strings:
                n_clients_per_round = int(strings[9])
        if "iteration" in strings:
            datestring = line.split(" : ")[0].replace(" ", "-")
            last_timestamp = pd.to_datetime(datestring, format="%a-%b-%d-%H:%M:%S-%Y")
            round_numbers.append(cnt)
            cnt += 1
            start_timestamp.append(previous_timestamp)
            previous_timestamp = last_timestamp
            end_timestamp.append(last_timestamp)
    # Creating dict to pass to pa.Table builder
    dict_for_table = {
        "rct": [
            (t1 - t0) for t0, t1 in zip(start_timestamp, end_timestamp, strict=False)
        ],
        "round": round_numbers,
        "n_clients_per_round": [n_clients_per_round] * len(round_numbers),
        "dataset": [dataset] * len(round_numbers),
        "framework": [framework] * len(round_numbers),
        "hardware_setting": [hardware_setting] * len(round_numbers),
        "is_dropped": [False] * len(round_numbers),
    }
    # Returning pa.Table
    return pa.Table.from_pydict(dict_for_table)

In [ ]:
results_table: pa.Table | None = None
for result_file in RESULTS_DIR.glob("*"):
    tmp_table: pa.Table | None = None
    filename = result_file.name
    try:
        if filename.startswith("log"):
            tmp_table = read_logs_fedscale(result_file)
        elif filename.startswith("flute"):
            tmp_table = read_logs_flute(result_file)
        else:
            tmp_table = read_logs_flwr(result_file)
        if tmp_table is not None:
            if results_table is None:
                results_table = tmp_table
            else:
                results_table = pa.concat_tables(
                    [results_table, tmp_table],
                )
    except Exception as e:
        print(f"Error while reading {filename}: {e}")

In [ ]:
# Drawing bar plots for each dataset, hardware setting = 1 and 100 clients per round
current_hardware_setting = 1
n_clients_per_round = 100
dfs: list[pd.DataFrame] = []
for framework in FRAMEWORKS:
    for dataset in DATASETS:
        random.seed(1337)
        tmp_table = (
            results_table.filter(pc.field("framework") == pc.scalar(framework))
            .filter(pc.field("dataset") == pc.scalar(dataset))
            .filter(pc.field("hardware_setting") == pc.scalar(current_hardware_setting))
            .filter(pc.field("n_clients_per_round") == pc.scalar(n_clients_per_round))
            .filter(pc.field("is_dropped") == pc.scalar(False))
        )
        try:
            if tmp_table.num_rows != 0:
                tmp_rct = (
                    tmp_table.column("rct")
                    .to_numpy()[1:]
                    .astype("timedelta64[s]")
                    .astype(float)
                )
                tmp_rct = random.choices(tmp_rct, k=N_TOTAL_ROUNDS)
                tmp_dict = {
                    "Round Completion Time [s]": tmp_rct,
                    "Round Completion Time [m]": np.array(tmp_rct) * (1 / 60),
                    "Round Completion Time [h]": np.array(tmp_rct) * (1 / 3600),
                    "Round Completion Time [d]": (
                        np.array(tmp_rct) * (1 / 3600) * (1 / 24)
                    ),
                    "Framework": [framework] * N_TOTAL_ROUNDS,
                    "Dataset": [dataset] * N_TOTAL_ROUNDS,
                }
                dfs.append(pd.DataFrame.from_dict(tmp_dict))
        except Exception as e:
            print(e)
textures = ["", ".", "x", "O", "/"]
colors = [
    sns.color_palette("colorblind")[3],
    sns.color_palette("colorblind")[1],
    sns.color_palette("colorblind")[2],
    sns.color_palette("colorblind")[0],
    sns.color_palette("colorblind")[7],
]
alphas = [0.4, 0.6, 0.8, 1.0, 1.2]
fig, ax = plt.subplots()
sns.barplot(
    data=pd.concat(dfs),
    x="Framework",
    y="Round Completion Time [h]",
    order=FRAMEWORKS,
    hue="Dataset",
    hue_order=DATASETS,
    errorbar=error_bar_fn_std_based,
    # errorbar=error_bar_fn_min_max,
    estimator=np.sum,
    ax=ax,
    edgecolor="black",
)
print(ax.patches)
for i, patch in enumerate(ax.patches):
    if i > 19:
        patch.set_facecolor("w")
        patch.set_hatch(textures[i % len(textures)])
        patch.set_alpha(1.0)
    else:
        patch.set_facecolor(colors[i % len(textures)])
        patch.set_hatch(textures[i // len(textures)])
        patch.set_alpha(alphas[i // len(alphas)])
y_axis_position = plt.gca().get_position().x0
plt.axhline(y=5, color="r", linestyle="--")
plt.text(y_axis_position - 1.17, 4.8, "5 hours", color="r", fontsize=11)
plt.axhline(y=24, color="r", linestyle="--")
plt.text(y_axis_position - 1.02, 23, "1 day", color="r", fontsize=11)
plt.axhline(y=24 * 7, color="r", linestyle="--")
plt.text(y_axis_position - 1.12, (24 * 7) - 12, "1 week", color="r", fontsize=11)
plt.axhline(y=24 * 14, color="r", linestyle="--")
plt.text(y_axis_position - 1.18, (24 * 14) - 14, "2 weeks", color="r", fontsize=11)
plt.axhline(y=24 * 30, color="r", linestyle="--")
plt.text(y_axis_position - 1.20, (24 * 30) - 30, "1 month", color="r", fontsize=11)
y_label = plt.ylabel("Total Time (log) [h]")
y_label.set_weight("bold")
plt.xlabel("")
ax.yaxis.set_label_coords(-0.1, 0.4)
plt.yscale("log")
plt.legend().set_title("")
legend_rect = plt.legend(loc="upper left").get_frame()
legend_rect.set_alpha(0)
plt.grid(axis="y")
plt.savefig("frameworks_singlenode.pdf", format="pdf", dpi=800, bbox_inches="tight")

In [ ]:
# Drawing bar plots for each dataset, hardware setting = 2 and 100 clients per round
current_hardware_setting = 2
n_clients_per_round = 100
dfs: list[pd.DataFrame] = []
for framework in FRAMEWORKS:
    for dataset in DATASETS:
        random.seed(1337)
        tmp_table = (
            results_table.filter(pc.field("framework") == pc.scalar(framework))
            .filter(pc.field("dataset") == pc.scalar(dataset))
            .filter(pc.field("hardware_setting") == pc.scalar(current_hardware_setting))
            .filter(pc.field("n_clients_per_round") == pc.scalar(n_clients_per_round))
            .filter(pc.field("is_dropped") == pc.scalar(False))
        )
        try:
            if tmp_table.num_rows != 0:
                tmp_rct = (
                    tmp_table.column("rct")
                    .to_numpy()[1:]
                    .astype("timedelta64[s]")
                    .astype(float)
                )
                tmp_rct = random.choices(tmp_rct, k=N_TOTAL_ROUNDS)
                tmp_dict = {
                    "Round Completion Time [s]": tmp_rct,
                    "Round Completion Time [m]": np.array(tmp_rct) * (1 / 60),
                    "Round Completion Time [h]": np.array(tmp_rct) * (1 / 3600),
                    "Framework": [framework] * N_TOTAL_ROUNDS,
                    "Dataset": [dataset] * N_TOTAL_ROUNDS,
                }
                dfs.append(pd.DataFrame.from_dict(tmp_dict))
        except Exception as e:
            print(e)
textures = ["", ".", "x", "O", "/"]
colors = [
    sns.color_palette("colorblind")[3],
    sns.color_palette("colorblind")[1],
    sns.color_palette("colorblind")[2],
    sns.color_palette("colorblind")[0],
    sns.color_palette("colorblind")[7],
]
alphas = [0.4, 0.6, 0.8, 1.0, 1.2]
fig, ax = plt.subplots()
sns.barplot(
    data=pd.concat(dfs),
    x="Framework",
    y="Round Completion Time [h]",
    order=FRAMEWORKS,
    hue="Dataset",
    hue_order=DATASETS,
    errorbar=error_bar_fn_std_based,
    # errorbar=error_bar_fn_min_max,
    estimator=np.sum,
    ax=ax,
    edgecolor="black",
)
print(ax.patches)
for i, patch in enumerate(ax.patches):
    if i > 19:
        patch.set_facecolor("w")
        patch.set_hatch(textures[i % len(textures)])
        patch.set_alpha(1.0)
    else:
        patch.set_facecolor(colors[i % len(textures)])
        patch.set_hatch(textures[i // len(textures)])
        patch.set_alpha(alphas[i // len(alphas)])
y_axis_position = plt.gca().get_position().x0
plt.axhline(y=5, color="r", linestyle="--")
plt.text(y_axis_position - 1.17, 4.8, "5 hours", color="r", fontsize=11)
plt.axhline(y=24, color="r", linestyle="--")
plt.text(y_axis_position - 1.0, 23, "1 day", color="r", fontsize=11)
plt.axhline(y=24 * 7, color="r", linestyle="--")
plt.text(y_axis_position - 1.1, (24 * 7) - 12, "1 week", color="r", fontsize=11)
plt.axhline(y=24 * 14, color="r", linestyle="--")
plt.text(y_axis_position - 1.18, (24 * 14) - 14, "2 weeks", color="r", fontsize=11)
y_label = plt.ylabel("Total Time (log) [h]")
y_label.set_weight("bold")
plt.xlabel("")
ax.yaxis.set_label_coords(-0.1, 0.45)
plt.yscale("log")
plt.legend().set_title("")
legend_rect = plt.legend(loc="upper left").get_frame()
legend_rect.set_alpha(0)
plt.grid(axis="y")
plt.savefig("frameworks_multinode.pdf", format="pdf", dpi=800, bbox_inches="tight")

In [ ]:
y_axis_label_coord = {
    "IC": [-0.12, 0.4],
    "SR": [-0.12, 0.4],
    "TG": [-0.1, 0.4],
    "MLM": [-0.1, 0.38],
}

texts_coord = {
    "IC": [0.86, 0.92, 0.96, 0.98, 1.01],
    "SR": [0.86, 0.92, 0.96, 0.98, 1.01],
    "TG": [0.86, 0.92, 0.96, 0.98, 1.01],
    "MLM": [0.78, 0.82, 0.84, 0.86, 0.88],
}

# Drawing bar plots for each dataset, hardware setting = 2 and 100 clients per round
current_hardware_setting = 2
for dataset in DATASETS:
    print(f"Dataset: {dataset}")
    dfs: list[pd.DataFrame] = []
    for framework in FRAMEWORKS:
        for n_clients_per_round, scale in SCALES[dataset].items():
            random.seed(1337)
            tmp_table = (
                results_table.filter(pc.field("framework") == pc.scalar(framework))
                .filter(pc.field("dataset") == pc.scalar(dataset))
                .filter(
                    pc.field("hardware_setting") == pc.scalar(current_hardware_setting)
                )
                .filter(
                    pc.field("n_clients_per_round") == pc.scalar(n_clients_per_round)
                )
                .filter(pc.field("is_dropped") == pc.scalar(False))
            )
            try:
                if tmp_table.num_rows != 0:
                    tmp_rct = (
                        tmp_table.column("rct")
                        .to_numpy()[1:]
                        .astype("timedelta64[s]")
                        .astype(float)
                    )
                    tmp_rct = random.choices(tmp_rct, k=N_TOTAL_ROUNDS)
                    tmp_dict = {
                        "Round Completion Time [s]": tmp_rct,
                        "Round Completion Time [m]": np.array(tmp_rct) * (1 / 60),
                        "Round Completion Time [h]": np.array(tmp_rct) * (1 / 3600),
                        "Framework": [framework] * N_TOTAL_ROUNDS,
                        "Dataset": [dataset] * N_TOTAL_ROUNDS,
                        "Scale (# clients per round)": [scale] * N_TOTAL_ROUNDS,
                    }
                    dfs.append(pd.DataFrame.from_dict(tmp_dict))
            except Exception as e:
                print(e)
    textures = ["", ".", "x", "O", "*"]
    colors = [
        sns.color_palette("colorblind")[3],
        sns.color_palette("colorblind")[1],
        sns.color_palette("colorblind")[2],
        sns.color_palette("colorblind")[0],
        sns.color_palette("colorblind")[7],
    ]
    alphas = [0.4, 0.6, 0.8, 1.0, 1.0]
    fig, ax = plt.subplots()
    sns.barplot(
        data=pd.concat(dfs),
        x="Scale (# clients per round)",
        y="Round Completion Time [h]",
        hue="Framework",
        errorbar=error_bar_fn_std_based,
        # errorbar=error_bar_fn_min_max,
        estimator=np.sum,
        edgecolor="black",
    )
    print(ax.patches)
    for i, patch in enumerate(ax.patches):
        if len(ax.patches) == 19 and i > 10:
            i += 1
        j, k = (3, 14) if dataset != "MLM" else (2, 9)
        if i > k:
            print(patch)
            patch.set_facecolor("w")
            patch.set_hatch(textures[i % len(textures)])
            patch.set_alpha(1.0)
        else:
            patch.set_facecolor(colors[i % j])
            patch.set_hatch(textures[i // j])
            patch.set_alpha(alphas[i // j])
    y_axis_position = plt.gca().get_position().x0
    plt.axhline(y=24, color="r", linestyle="--")
    plt.text(
        y_axis_position - texts_coord[dataset][0], 23, "1 day", color="r", fontsize=11
    )
    plt.axhline(y=24 * 7, color="r", linestyle="--")
    plt.text(
        y_axis_position - texts_coord[dataset][1],
        (24 * 7) - 12,
        "1 week",
        color="r",
        fontsize=11,
    )
    plt.axhline(y=24 * 14, color="r", linestyle="--")
    plt.text(
        y_axis_position - texts_coord[dataset][2],
        (24 * 14) - 14,
        "2 weeks",
        color="r",
        fontsize=11,
    )
    if dataset == "SR" or dataset == "MLM":
        plt.axhline(y=24 * 30, color="r", linestyle="--")
        plt.text(
            y_axis_position - texts_coord[dataset][3],
            (24 * 30) - 14,
            "1 month",
            color="r",
            fontsize=11,
        )
    plt.axhline(y=24 * 60, color="r", linestyle="--")
    plt.text(
        y_axis_position - texts_coord[dataset][4],
        (24 * 60) - 14,
        "2 months",
        color="r",
        fontsize=11,
    )
    if dataset == "TG":
        plt.text(0.90, 79, r"$(\star)$", color="red", fontsize=15)
        plt.text(1.90, 795, r"$(\star)$", color="red", fontsize=15)
    if dataset == "IC":
        for i in range(10):
            plt.text(2.078, 15 * (2**i), r"$(\star)$", color="red", fontsize=12)

    y_label = plt.ylabel("Total Time [h]")
    y_label.set_weight("bold")
    ax.yaxis.set_label_coords(
        y_axis_label_coord[dataset][0],
        y_axis_label_coord[dataset][1],
    )
    plt.yscale("log")
    plt.xlabel("")
    legend = plt.legend(handleheight=1.5)
    legend.set_title("")
    legend.get_frame().set_alpha(0)
    plt.grid(axis="y")
    plt.savefig(
        f"scalability_{dataset.lower()}.pdf", format="pdf", dpi=800, bbox_inches="tight"
    )
    plt.show()

In [ ]:
# Drawing bar plots for each dataset, hardware setting = 2 and 100 clients per round
current_hardware_setting = 2
n_clients_per_round = 1000
dfs: list[pd.DataFrame] = []
for framework in FRAMEWORKS:
    for dataset in DATASETS:
        random.seed(1337)
        tmp_table = (
            results_table.filter(pc.field("framework") == pc.scalar(framework))
            .filter(pc.field("dataset") == pc.scalar(dataset))
            .filter(pc.field("hardware_setting") == pc.scalar(current_hardware_setting))
            .filter(pc.field("n_clients_per_round") == pc.scalar(n_clients_per_round))
            .filter(pc.field("is_dropped") == pc.scalar(False))
        )
        try:
            if tmp_table.num_rows != 0:
                tmp_rct = (
                    tmp_table.column("rct")
                    .to_numpy()[1:]
                    .astype("timedelta64[s]")
                    .astype(float)
                )
                tmp_rct = random.choices(tmp_rct, k=N_TOTAL_ROUNDS)
                tmp_dict = {
                    "Round Completion Time [s]": tmp_rct,
                    "Round Completion Time [m]": np.array(tmp_rct) * (1 / 60),
                    "Round Completion Time [h]": np.array(tmp_rct) * (1 / 3600),
                    "Framework": [framework] * N_TOTAL_ROUNDS,
                    "Dataset": [dataset] * N_TOTAL_ROUNDS,
                }
                dfs.append(pd.DataFrame.from_dict(tmp_dict))
        except Exception as e:
            print(e)
fig, ax = plt.subplots()
textures = ["", ".", "x", "O", "/"]
colors = [
    sns.color_palette("colorblind")[3],
    sns.color_palette("colorblind")[1],
    sns.color_palette("colorblind")[2],
    sns.color_palette("colorblind")[0],
    sns.color_palette("colorblind")[7],
]
alphas = [0.4, 0.6, 0.8, 1.0, 1.2]
sns.barplot(
    data=pd.concat(dfs),
    x="Framework",
    y="Round Completion Time [h]",
    order=FRAMEWORKS,
    hue="Dataset",
    hue_order=DATASETS,
    errorbar=error_bar_fn_std_based,
    # errorbar=error_bar_fn_min_max,
    estimator=np.sum,
    edgecolor="black",
)
print(ax.patches)
for i, patch in enumerate(ax.patches):
    if i > 19:
        patch.set_facecolor("w")
        patch.set_hatch(textures[i % len(textures)])
        patch.set_alpha(1.0)
    else:
        patch.set_facecolor(colors[i % len(textures)])
        patch.set_hatch(textures[i // len(textures)])
        patch.set_alpha(alphas[i // len(alphas)])

y_axis_position = plt.gca().get_position().x0
plt.axhline(y=24, color="r", linestyle="--")
plt.text(y_axis_position - 1.0, 23, "1 day", color="r", fontsize=11)
plt.axhline(y=24 * 7, color="r", linestyle="--")
plt.text(y_axis_position - 1.1, (24 * 7) - 12, "1 week", color="r", fontsize=11)
plt.axhline(y=24 * 14, color="r", linestyle="--")
plt.text(y_axis_position - 1.16, (24 * 14) - 14, "2 weeks", color="r", fontsize=11)
y_label = plt.ylabel("Total Time (log) [h]")
y_label.set_weight("bold")
plt.yscale("log")
ax.yaxis.set_label_coords(-0.08, 0.80)
plt.xlabel("")
plt.legend().set_title("")
plt.legend().get_frame().set_alpha(0)
plt.grid(axis="y")
plt.savefig("the_fig_one.pdf", format="pdf", dpi=800, bbox_inches="tight")

In [ ]:
y_axis_label_coord = {
    "IC": [-0.12, 0.4],
    "SR": [-0.12, 0.4],
    "TG": [-0.1, 0.4],
    "MLM": [-0.1, 0.36],
}

texts_coord = {
    "SR": [0.93, 1.0, 1.04, 1.08, 1.12],
    "MLM": [0.86, 0.92, 0.96, 0.98, 1.01],
}

inner_scales = {
    "SR": {
        100: "Medium",
        1_000: "Large",
        2_000: "Very Large",
        10_000: "Ultra Large",
    },
    "MLM": {
        100: "Medium",
        1_000: "Large",
        10_000: "Very Large",
    },
}
map_patches_1 = {
    0: 0,
    1: 0,
    2: 0,
    3: 0,
    4: 1,
    5: 1,
    6: 1,
    7: 2,
    8: 2,
    9: 2,
    10: 3,
    11: 3,
    12: 3,
    13: 4,
    14: 4,
    15: 4,
    16: 0,
    17: 1,
    18: 2,
    19: 3,
    20: 4,
}
map_patches_2 = {
    0: 0,
    1: 0,
    2: 0,
    3: 1,
    4: 1,
    5: 2,
    6: 2,
    7: 3,
    8: 3,
    9: 4,
    10: 4,
    11: 0,
    12: 1,
    13: 2,
    14: 3,
    15: 4,
}
map_colors_1 = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 0,
    5: 1,
    6: 2,
    7: 0,
    8: 1,
    9: 2,
    10: 0,
    11: 1,
    12: 2,
    13: 0,
    14: 1,
    15: 2,
}
map_colors_2 = {
    0: 0,
    1: 1,
    2: 2,
    3: 0,
    4: 1,
    5: 0,
    6: 1,
    7: 0,
    8: 1,
    9: 0,
    10: 1,
}

# Drawing bar plots for each dataset, hardware setting = 2 and 100 clients per round
current_hardware_setting = 2
for dataset in ["SR", "MLM"]:
    print(f"Dataset: {dataset}")
    dfs: list[pd.DataFrame] = []
    for framework in FRAMEWORKS:
        for n_clients_per_round, scale in inner_scales[dataset].items():
            random.seed(1337)
            tmp_table = (
                results_table.filter(pc.field("framework") == pc.scalar(framework))
                .filter(pc.field("dataset") == pc.scalar(dataset))
                .filter(
                    pc.field("hardware_setting") == pc.scalar(current_hardware_setting)
                )
                .filter(
                    pc.field("n_clients_per_round") == pc.scalar(n_clients_per_round)
                )
                .filter(pc.field("is_dropped") == pc.scalar(False))
            )
            try:
                if tmp_table.num_rows != 0:
                    tmp_rct = (
                        tmp_table.column("rct")
                        .to_numpy()[1:]
                        .astype("timedelta64[s]")
                        .astype(float)
                    )
                    tmp_rct = random.choices(tmp_rct, k=N_TOTAL_ROUNDS)
                    tmp_dict = {
                        "Round Completion Time [s]": tmp_rct,
                        "Round Completion Time [m]": np.array(tmp_rct) * (1 / 60),
                        "Round Completion Time [h]": np.array(tmp_rct) * (1 / 3600),
                        "Framework": [framework] * N_TOTAL_ROUNDS,
                        "Dataset": [dataset] * N_TOTAL_ROUNDS,
                        "Scale (# clients per round)": [scale] * N_TOTAL_ROUNDS,
                    }
                    dfs.append(pd.DataFrame.from_dict(tmp_dict))
            except Exception as e:
                print(e)
    textures = ["", ".", "x", "O", "*"]
    colors = [
        sns.color_palette("colorblind")[3],
        sns.color_palette("colorblind")[1],
        sns.color_palette("colorblind")[2],
        sns.color_palette("colorblind")[0],
        sns.color_palette("colorblind")[7],
    ]
    alphas = [0.4, 0.6, 0.8, 1.0, 1.0]
    fig, ax = plt.subplots()
    sns.barplot(
        data=pd.concat(dfs),
        x="Scale (# clients per round)",
        y="Round Completion Time [h]",
        hue="Framework",
        errorbar=error_bar_fn_std_based,
        # errorbar=error_bar_fn_min_max,
        estimator=np.sum,
        edgecolor="black",
    )
    print(ax.patches)
    for i, patch in enumerate(ax.patches):
        k = 15 if dataset != "MLM" else 10
        map_patches = map_patches_1 if dataset != "MLM" else map_patches_2
        map_colors = map_colors_1 if dataset != "MLM" else map_colors_2
        patch.set_hatch(textures[map_patches[i]])
        if i > k:
            patch.set_facecolor("w")
            patch.set_alpha(1.0)
        else:
            patch.set_facecolor(colors[map_colors[i]])
            patch.set_alpha(alphas[map_patches[i]])
    y_axis_position = plt.gca().get_position().x0
    plt.axhline(y=24, color="r", linestyle="--")
    plt.text(
        y_axis_position - texts_coord[dataset][0], 23, "1 day", color="r", fontsize=11
    )
    plt.axhline(y=24 * 7, color="r", linestyle="--")
    plt.text(
        y_axis_position - texts_coord[dataset][1],
        (24 * 7) - 12,
        "1 week",
        color="r",
        fontsize=11,
    )
    plt.axhline(y=24 * 14, color="r", linestyle="--")
    plt.text(
        y_axis_position - texts_coord[dataset][2],
        (24 * 14) - 14,
        "2 weeks",
        color="r",
        fontsize=11,
    )
    if dataset == "SR" or dataset == "MLM":
        plt.axhline(y=24 * 30, color="r", linestyle="--")
        plt.text(
            y_axis_position - texts_coord[dataset][3],
            (24 * 30) - 14,
            "1 month",
            color="r",
            fontsize=11,
        )
    plt.axhline(y=24 * 60, color="r", linestyle="--")
    plt.text(
        y_axis_position - texts_coord[dataset][4],
        (24 * 60) - 14,
        "2 months",
        color="r",
        fontsize=11,
    )
    if dataset == "TG":
        plt.text(0.90, 79, r"$(\star)$", color="red", fontsize=15)
        plt.text(1.90, 795, r"$(\star)$", color="red", fontsize=15)
    if dataset == "IC":
        for i in range(10):
            plt.text(2.078, 15 * (2**i), r"$(\star)$", color="red", fontsize=12)

    y_label = plt.ylabel("Total Time [h]")
    y_label.set_weight("bold")
    ax.yaxis.set_label_coords(
        y_axis_label_coord[dataset][0],
        y_axis_label_coord[dataset][1],
    )
    plt.yscale("log")
    plt.xlabel("")
    legend = plt.legend(handleheight=1.3, loc="upper left")
    legend.set_title("")
    legend.get_frame().set_alpha(0)
    plt.grid(axis="y")
    plt.savefig(
        f"scalability_appendix_{dataset.lower()}.pdf",
        format="pdf",
        dpi=800,
        bbox_inches="tight",
    )
    plt.show()